# Native polynomial QSVT with phase fitting

Retain the original degree-five odd polynomial, seed 1 and normalization on 500 points. Fit new angles with the existing native local optimizer, then execute the native qfunc. No cloud optimizer-equivalence, hardware or generic phase-solver claim is made.


In [ ]:
import torch

from examples.synthesis.functions.qsvt.overview.main import (
    Workflow,
    fit_phases,
    plot_results,
    polynomial,
    projected_values,
    source_coefficients,
)

coefficients = source_coefficients()
fit = fit_phases(coefficients)
print("Chebyshev coefficients:", coefficients.tolist())
print("Phase fit:", fit)


## Quantum composition

The existing native qfunc allocates state and reference registers, comparator indicator and QSVT auxiliary. It composes Hadamards, reversible comparison XOR, projected phases and within/apply. Newly fitted angles are classical synthesis-time inputs. The scalar model uses half of each native projector rotation angle.


In [ ]:
from examples.synthesis.functions.qsvt.main import project_zero
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QBit,
    QNum,
    allocate,
    apply,
    hadamard_transform,
    qfunc,
    within,
)


@qfunc
def qsvt_program(
    phases: list[float],
    state: Output[QNum],
    ref: Output[QNum],
    ind: Output[QBit],
    qsvt_aux: Output[QBit],
) -> None:
    allocate(4, state)
    allocate(4, ref)
    allocate(ind)
    allocate(qsvt_aux)
    hadamard_transform(state)
    H(qsvt_aux)
    project_zero(phases[0], qsvt_aux, ref)
    for step in range(2):
        with within():
            hadamard_transform(ref)
            ind ^= state <= ref
        with apply():
            project_zero(phases[2 * step + 1], qsvt_aux, ind)
        project_zero(phases[2 * step + 2], qsvt_aux, ref)
    hadamard_transform(ref)
    ind ^= state <= ref
    project_zero(phases[5], qsvt_aux, ind)
    H(qsvt_aux)


program = synth.synthesize(
    qsvt_program,
    classical_inputs={"phases": list(fit.phases)},
    max_qubits=16,
    max_workspace=6,
    memory_limit_bytes=8 * 1024**2,
)
print(program.report["resources"])


## Actual named readout and original projection

For this odd-degree example, select ind=0 and qsvt_aux=0 and sum over ref. Multiply the square root of the joint probabilities by four. Named outputs avoid wire numbers and bitstring decoding. No fixed-phase fallback is used.


In [ ]:
points = (torch.arange(16, dtype=torch.float64) / 16).sqrt()
measured = projected_values(program)
target = polynomial(coefficients, points).abs()
torch.testing.assert_close(measured, target, atol=1e-7, rtol=0)
workflow = Workflow(fit, program, points, measured, target)
print("Measured:", measured.tolist())
print("Target:", target.tolist())
print("Maximum error:", float((measured - target).abs().max()))


## Polynomial comparison

The source uses 0.02 tolerance; this workflow additionally requires 1e-7. Different fitted phase branches can change complementary quantum amplitudes. Full-state Circuit comparison therefore uses these newly fitted phases; the frozen original phase example remains separately validated. Fit-grid checks do not provide an analytic uniform-error certificate.


In [ ]:
figure = plot_results(workflow)
